In [37]:
# ------------------------------------------------------------
# C2 (v2c). River Discharge vs. Stations crossing — version-aware, ID-stable
# + inherit ALL reach attributes (WSE/width/slope/area/… + discharges & flags)
# + robust CRID extraction, streaming read, AOI prefilter, D-over-C precedence
#
# Outputs (BASE/output/1_Crossing_v2):
#   - combined_timeseries.csv              (one row per station×reach_id_canon×obs_time; D-over-C)
#   - per_version_C.csv, per_version_D.csv (raw per-version, ALL attributes)
#   - id_resolver_summary.csv
#   - unmapped_rows.csv                    (C rows that couldn't map to D; for review)
#   - dedupe_log.csv                       (rows dropped during D-over-C arbitration)
#   - combined_reaches.gpkg:
#       • combined_reaches        (lean layer: keys + provenance + geometry)
#       • combined_reaches_full   (FULL attributes of winning row + geometry)
#
# Version: 1.5 — 2025-09-04 (v2c: carry ALL attributes + full GPKG layer)
# ------------------------------------------------------------

In [38]:
# =========================
# Block 0 — Imports
# =========================
import os, re, glob
from pathlib import Path

import pandas as pd
import geopandas as gpd
import fiona
from shapely.geometry import box, shape

In [39]:
# =========================
# Block 1 — User Settings
# =========================
BASE = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge")

# Inputs
PATH_STATIONS = BASE / r"input\Stations\PatriciaTEAMS\Round_3_Process\All_stations_Rwanda_WaterPortal.shp"  # AOI only
PATH_SWOT_BASE = BASE / r"input\SWOT_L2_HR_RiverSP"

RIVERSP_DIRS = {
    "C": PATH_SWOT_BASE / "versionC" / "reach",
    "D": PATH_SWOT_BASE / "versionD" / "reach",
}
REACH_GLOB = "**/*.shp"  # reach layers

# Crosswalks generated in C3
DIAG_DIR = BASE / r"output\diagnosis"
PATH_CROSSWALK = DIAG_DIR / "version_id_crosswalk.csv"
PATH_OVERLAP_MATCHES = DIAG_DIR / "overlap_matches.csv"  # fallback if crosswalk missing

# Outputs
OUT_DIR = BASE / r"output\1_Crossing_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_COMBINED_CSV = OUT_DIR / "combined_timeseries.csv"
OUT_PER_C = OUT_DIR / "per_version_C.csv"
OUT_PER_D = OUT_DIR / "per_version_D.csv"
OUT_RESOLVER_SUMMARY = OUT_DIR / "id_resolver_summary.csv"
OUT_UNMAPPED = OUT_DIR / "unmapped_rows.csv"
OUT_DEDUPE = OUT_DIR / "dedupe_log.csv"
OUT_GPKG = OUT_DIR / "combined_reaches.gpkg"

# AOI / performance tuning
STATION_PREFILTER_BUFFER_KM = 50  # buffer for AOI (around station points)
READ_WITHIN_AOI_ONLY = True       # set False to read all files (slower)

# Toggles
WRITE_GPKG = True
DEBUG = True

In [40]:
# =========================
# Block 2 — Utilities
# =========================
def log(msg):
    if DEBUG:
        print(msg)

def ensure_dir(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)

def classify_version_from_path(path_str):
    p = path_str.replace("\\", "/").lower()
    if "/versionc/" in p: return "C"
    if "/versiond/" in p: return "D"
    # fallback: try CRID letter in filename
    m = re.search(r"P[GI]([CD])[A-Z0-9_]+", os.path.basename(path_str), flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return "?"

# robust CRID extraction (e.g., PGC0_01, PIC2_01, PID0_03)
CRID_REGEX = re.compile(r"P[GI][CD][A-Z0-9_]+", re.IGNORECASE)
def extract_crid_from_text(text):
    if not text:
        return ""
    m = CRID_REGEX.search(text)
    return m.group(0) if m else ""

def list_reach_files(root: Path):
    if not root.exists():
        return []
    return sorted(glob.glob(str(root / REACH_GLOB), recursive=True))

def geoms_union_safe(geo_series):
    """Use union_all() if available, else unary_union (compat shim)."""
    return geo_series.union_all() if hasattr(geo_series, "union_all") else geo_series.unary_union

def build_aoi_from_stations(path_stations: Path, buffer_km=50):
    if not path_stations.exists():
        log(f"[WARN] Stations file not found: {path_stations}. AOI disabled.")
        return None, None
    s = gpd.read_file(path_stations)
    if s.crs is None:
        s = s.set_crs(4326)
    # pick a rough UTM by centroid for metric buffering
    cxy = geoms_union_safe(s.to_crs(4326).geometry).centroid
    lon, lat = float(cxy.x), float(cxy.y)
    zone = int((lon + 180)//6) + 1
    mcrs = f"EPSG:{32600+zone if lat>=0 else 32700+zone}"
    s_m = s.to_crs(mcrs)
    buf_m = buffer_km * 1000.0
    aoi_m = geoms_union_safe(s_m.buffer(buf_m))
    aoi_4326 = gpd.GeoSeries([aoi_m], crs=mcrs).to_crs(4326).iloc[0]
    bbox = aoi_4326.envelope.bounds
    return aoi_4326, bbox

def preselect_files_by_aoi(files, bbox_4326):
    if bbox_4326 is None:
        return files
    AOI = box(*bbox_4326)
    out = []
    for f in files:
        try:
            with fiona.open(f) as src:
                minx, miny, maxx, maxy = src.bounds
            if box(minx, miny, maxx, maxy).intersects(AOI):
                out.append(f)
        except Exception as e:
            log(f"[WARN] bounds fail {f}: {e}")
    return out

In [41]:
# =========================
# Block 3 — Load crosswalk (and build station-aware maps)
# =========================
if not PATH_CROSSWALK.exists():
    # Fallback: derive from overlap_matches.csv if crosswalk missing
    if not PATH_OVERLAP_MATCHES.exists():
        raise FileNotFoundError(
            f"Crosswalk not found at {PATH_CROSSWALK} and fallback {PATH_OVERLAP_MATCHES} missing.\n"
            "Run the C3 diagnosis first to create version_id_crosswalk.csv."
        )
    mm = pd.read_csv(PATH_OVERLAP_MATCHES)
    counts = (mm.groupby(["station_id","station_reach_id_ref","version_hint","matched_reach_id"])
                .size().rename("n").reset_index())

    def top(grp):
        if grp.empty: return None, 0, 0, 0.0
        s = grp.groupby("matched_reach_id")["n"].sum().sort_values(ascending=False)
        rid_top = s.index[0]; n_top = int(s.iloc[0]); tot = int(s.sum()); share = n_top/tot if tot>0 else 0.0
        return rid_top, n_top, tot, share

    rows=[]
    for (sid, rid), g in counts.groupby(["station_id","station_reach_id_ref"]):
        gC = g[g["version_hint"]=="C"]; gD = g[g["version_hint"]=="D"]
        c_id,c_n,c_tot,c_share = top(gC)
        d_id,d_n,d_tot,d_share = top(gD)
        rows.append({"station_id":sid,"station_reach_id_ref":rid,
                     "C_top_id":c_id,"C_top_n":c_n,"C_total":c_tot,"C_share":round(c_share,3),
                     "D_top_id":d_id,"D_top_n":d_n,"D_total":d_tot,"D_share":round(d_share,3)})
    crosswalk = pd.DataFrame(rows)
else:
    crosswalk = pd.read_csv(PATH_CROSSWALK)

crosswalk["station_id"] = crosswalk["station_id"].astype(str)
for col in ["station_reach_id_ref","C_top_id","D_top_id"]:
    if col in crosswalk.columns:
        crosswalk[col] = crosswalk[col].astype(str)

# Station-aware C→D map (only where both sides exist)
cw_change = crosswalk.dropna(subset=["C_top_id","D_top_id"]).copy()
c_to_d_by_station = { (r.station_id, r.C_top_id): r.D_top_id for r in cw_change.itertuples() }

# Reverse maps: reach_id → {stations}
d_id_to_station, c_id_to_station = {}, {}
for r in crosswalk.itertuples():
    if pd.notna(r.D_top_id): d_id_to_station.setdefault(str(r.D_top_id), set()).add(str(r.station_id))
    if pd.notna(r.C_top_id): c_id_to_station.setdefault(str(r.C_top_id), set()).add(str(r.station_id))

# Target ID sets per version (for fast filtering)
TARGET_IDS = {
    "D": set(str(x) for x in crosswalk["D_top_id"].dropna().unique()),
    "C": set(str(x) for x in crosswalk["C_top_id"].dropna().unique()),
}

In [42]:
# =========================
# Block 4 — Build AOI and index files (union_all-safe)
# =========================
aoi_poly, aoi_bbox = build_aoi_from_stations(PATH_STATIONS, buffer_km=STATION_PREFILTER_BUFFER_KM)
if aoi_bbox:
    log(f"AOI bbox (deg): {aoi_bbox}")
else:
    log("[INFO] AOI disabled; will scan all files (slower).")

files_C = list_reach_files(RIVERSP_DIRS["C"])
files_D = list_reach_files(RIVERSP_DIRS["D"])
log(f"Found reach files — C: {len(files_C)}, D: {len(files_D)}")

if READ_WITHIN_AOI_ONLY and aoi_bbox:
    files_C = preselect_files_by_aoi(files_C, aoi_bbox)
    files_D = preselect_files_by_aoi(files_D, aoi_bbox)
    log(f"After AOI prefilter — C: {len(files_C)}, D: {len(files_D)}")

AOI bbox (deg): (-2.003782771619399, -3.264747324225748, 31.22948709026701, 30.044209012087663)
Found reach files — C: 722, D: 156
After AOI prefilter — C: 714, D: 155


In [43]:
# =========================
# Block 5 — Scan files (STREAMING), carry ALL properties
# =========================
RESERVED = {
    "station_id","reach_id","obs_time","source_file","source_dir",
    "source_crid","source_version","sword_version_assumed","geometry"
}

def scan_files_streaming(files, version_label, target_ids, bbox):
    rows = []
    total = len(files)
    for i, f in enumerate(files, 1):
        if i % 50 == 0 or i == 1 or i == total:
            log(f"[{version_label}] scanning {i}/{total}: {os.path.basename(f)}")

        try:
            with fiona.open(f) as src:
                use_bbox = bbox  # RiverSP reaches are EPSG:4326; bbox okay
                src_file = os.path.basename(f)
                src_dir  = os.path.dirname(f)
                version  = classify_version_from_path(f)
                sword_version = "v17b" if version == "D" else ("v16" if version == "C" else "")
                src_crid = extract_crid_from_text(src_file) or extract_crid_from_text(src_dir)

                iterator = src.filter(bbox=use_bbox) if use_bbox else src
                for feat in iterator:
                    props = feat.get("properties") or {}
                    rid = str(props.get("reach_id", ""))

                    if rid not in target_ids:
                        continue

                    # collect ALL properties, excluding our reserved names to avoid collisions
                    extras = {k: v for k, v in props.items() if k not in RESERVED}

                    # normalize obs_time (but keep original time fields in extras)
                    obs_time = props.get("time_str") or props.get("time_tai") or props.get("time") or None

                    stations = (d_id_to_station.get(rid, set()) if version_label == "D"
                                else c_id_to_station.get(rid, set()))
                    if not stations:
                        stations = {None}

                    geom = shape(feat["geometry"]) if feat.get("geometry") else None

                    for sid in stations:
                        row = {
                            "station_id": str(sid) if sid is not None else None,
                            "reach_id": rid,
                            "obs_time": str(obs_time) if obs_time is not None else None,
                            "source_file": src_file,
                            "source_dir": src_dir,
                            "source_crid": src_crid,
                            "source_version": version,
                            "sword_version_assumed": sword_version,
                            "geometry": geom
                        }
                        row.update(extras)
                        rows.append(row)

        except Exception as e:
            log(f"[WARN] streaming read fail {f}: {e}")
            continue

    cols_base = ["station_id","reach_id","obs_time","source_file","source_dir",
                 "source_crid","source_version","sword_version_assumed","geometry"]
    if not rows:
        return gpd.GeoDataFrame(columns=cols_base, geometry="geometry", crs="EPSG:4326")

    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    return gdf

log("Scanning Version C files…")
gC = scan_files_streaming(files_C, "C", TARGET_IDS["C"], aoi_bbox if (READ_WITHIN_AOI_ONLY and aoi_bbox) else None)
log(f"Kept rows (C): {len(gC)}")

log("Scanning Version D files…")
gD = scan_files_streaming(files_D, "D", TARGET_IDS["D"], aoi_bbox if (READ_WITHIN_AOI_ONLY and aoi_bbox) else None)
log(f"Kept rows (D): {len(gD)}")

# ---- Sanity checks (printing only) ----
print("\n--- Sanity checks after Block 5 ---")
print("C rows:", len(gC), " | D rows:", len(gD))
if not gC.empty:
    print("C unknown IDs:", (set(gC["reach_id"]) - TARGET_IDS["C"]) or "None")
if not gD.empty:
    print("D unknown IDs:", (set(gD["reach_id"]) - TARGET_IDS["D"]) or "None")
if not gC.empty:
    print("\nTop stations in C:", gC["station_id"].value_counts().head())
if not gD.empty:
    print("\nTop stations in D:", gD["station_id"].value_counts().head())


Scanning Version C files…
[C] scanning 1/714: SWOT_L2_HR_RiverSP_Reach_001_152_AF_20230726T152340_20230726T152349_PGC0_01.shp
[C] scanning 50/714: SWOT_L2_HR_RiverSP_Reach_004_305_AF_20231002T164441_20231002T164451_PGC0_01.shp
[C] scanning 100/714: SWOT_L2_HR_RiverSP_Reach_006_402_AF_20231116T212627_20231116T212629_PGC0_01.shp
[C] scanning 150/714: SWOT_L2_HR_RiverSP_Reach_007_458_AF_20231209T181225_20231209T181234_PGC0_01.shp
[C] scanning 200/714: SWOT_L2_HR_RiverSP_Reach_009_068_AF_20240106T131757_20240106T131801_PGC0_01.shp
[C] scanning 250/714: SWOT_L2_HR_RiverSP_Reach_010_318_AF_20240205T082436_20240205T082447_PIC0_01.shp
[C] scanning 300/714: SWOT_L2_HR_RiverSP_Reach_012_471_AF_20240323T130631_20240323T130632_PIC0_01.shp
[C] scanning 350/714: SWOT_L2_HR_RiverSP_Reach_015_180_AF_20240514T175030_20240514T175037_PIC0_01.shp
[C] scanning 400/714: SWOT_L2_HR_RiverSP_Reach_017_249_AF_20240627T222943_20240627T222951_PIC0_01.shp
[C] scanning 450/714: SWOT_L2_HR_RiverSP_Reach_019_471_AF_2

In [50]:
# =========================
# Block 6 — Canonicalize IDs, add crosswalk status
# =========================
def add_id_resolution(gdf):
    df = gdf.copy()
    if df.empty:
        df["reach_id_raw"] = []
        df["reach_id_canon"] = []
        df["crosswalk_status"] = []
        return df

    df["station_id"] = df["station_id"].astype(str)
    df["reach_id_raw"] = df["reach_id"].astype(str)

    canon = []
    status = []
    for r in df.itertuples():
        if r.source_version == "D":
            canon.append(r.reach_id_raw)
            status.append("stable")
        elif r.source_version == "C":
            key = (str(r.station_id), str(r.reach_id_raw))
            if key in c_to_d_by_station:
                canon.append(c_to_d_by_station[key])
                status.append("C→D")
            else:
                canon.append(r.reach_id_raw)
                status.append("unmapped")
        else:
            canon.append(r.reach_id_raw)
            status.append("unknown")
    df["reach_id_canon"] = canon
    df["crosswalk_status"] = status
    return df

gC_res = add_id_resolution(gC)
gD_res = add_id_resolution(gD)

In [51]:
# =========================
# Block 7 — Build unique combined table (D-over-C) WITH ALL attributes
# =========================
key_cols = ["station_id","reach_id_canon","obs_time"]

# We'll drop geometry for the tabular outputs; keep ALL other columns
def drop_geom(df):
    return pd.DataFrame(df.drop(columns="geometry"))

gC_tbl = drop_geom(gC_res).copy()
gD_tbl = drop_geom(gD_res).copy()
gC_tbl["version_rank"] = 1
gD_tbl["version_rank"] = 2

combined_full = pd.concat([gC_tbl, gD_tbl], ignore_index=True)
combined_full = combined_full.sort_values(key_cols + ["version_rank"], ascending=[True, True, True, False])

# Exactly one row per key (winner keeps its own full attribute set)
dedupe_keep = combined_full.drop_duplicates(subset=key_cols, keep="first").copy()

# Log the dropped rows for traceability
dropped_mask = ~combined_full.index.isin(dedupe_keep.index)
dedupe_log = combined_full.loc[dropped_mask].copy()
dedupe_log["dedupe_reason"] = "prefer_D_if_available"
ensure_dir(OUT_DEDUPE); dedupe_log.to_csv(OUT_DEDUPE, index=False)

# Per-version CSVs (ALL attributes)
gC_tbl.drop(columns=["version_rank"]).to_csv(OUT_PER_C, index=False)
gD_tbl.drop(columns=["version_rank"]).to_csv(OUT_PER_D, index=False)

# Final combined time series (winner rows only; ALL attributes)
df_combined = dedupe_keep.drop(columns=["version_rank"]).copy()
df_combined.to_csv(OUT_COMBINED_CSV, index=False)
log(f"Wrote: {OUT_COMBINED_CSV}")
log(f"Wrote: {OUT_PER_C}")
log(f"Wrote: {OUT_PER_D}")

Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\combined_timeseries.csv
Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\per_version_C.csv
Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\per_version_D.csv


In [52]:
# =========================
# Block 8 — Resolver summary & unmapped report
# =========================
def summarize_counts(df, label: str):
    if df is None or df.empty:
        return pd.Series(dtype=int, name=f"n_rows_{label}")
    return df.groupby("station_id").size().rename(f"n_rows_{label}")

sum_C = summarize_counts(gC_tbl, "C")
sum_D = summarize_counts(gD_tbl, "D")
sum_comb = summarize_counts(df_combined, "combined")

resolver = crosswalk[["station_id","station_reach_id_ref","C_top_id","D_top_id","C_share","D_share"]].copy()
resolver = resolver.merge(sum_C, on="station_id", how="left")
resolver = resolver.merge(sum_D, on="station_id", how="left")
resolver = resolver.merge(sum_comb, on="station_id", how="left")
for c in ["n_rows_C","n_rows_D","n_rows_combined"]:
    resolver[c] = resolver[c].fillna(0).astype(int)
resolver.to_csv(OUT_RESOLVER_SUMMARY, index=False)
log(f"Wrote: {OUT_RESOLVER_SUMMARY}")

# Any unmapped C rows? (keep their attributes for inspection)
unmapped = pd.DataFrame()
if not gC_tbl.empty:
    mask_unmapped = (gC_tbl["source_version"]=="C") & (gC_tbl["reach_id_canon"]==gC_tbl["reach_id_raw"])
    unmapped = gC_tbl.loc[mask_unmapped].copy()
if not unmapped.empty:
    unmapped.to_csv(OUT_UNMAPPED, index=False)
    log(f"[INFO] Unmapped C rows written: {OUT_UNMAPPED}")
else:
    log("[INFO] No unmapped C rows.")

Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\id_resolver_summary.csv
[INFO] Unmapped C rows written: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\unmapped_rows.csv


In [53]:
# =========================
# Block 9 — GeoPackage (two layers: lean + FULL attributes), memory-safe
# =========================
if WRITE_GPKG:
    key = ["station_id","reach_id_canon","obs_time"]

    # Version-labelled geometry tables
    gC_k = gC_res[key + ["geometry"]].copy()
    gC_k = gpd.GeoDataFrame(gC_k, geometry="geometry", crs="EPSG:4326").rename(columns={"geometry":"geom_C"})
    gD_k = gD_res[key + ["geometry"]].copy()
    gD_k = gpd.GeoDataFrame(gD_k, geometry="geometry", crs="EPSG:4326").rename(columns={"geometry":"geom_D"})

    # Ensure one geometry per key (pick first)
    def make_unique_per_key(gdf, geom_col):
        uniq = (gdf.dropna(subset=[geom_col])
                    .drop_duplicates(subset=key, keep="first")
                    .copy())
        if uniq.duplicated(subset=key).any():
            uniq = (uniq.groupby(key, as_index=False)
                        .agg({geom_col: "first"}))
        return uniq

    gD_single = make_unique_per_key(gD_k, "geom_D")
    gC_single = make_unique_per_key(gC_k, "geom_C")

    # Keys present in the combined (winner) table
    kept_keys = df_combined[key].drop_duplicates().copy()

    # Join D geometry then C (fallback); choose D if present
    base = kept_keys.merge(gD_single, on=key, how="left").merge(gC_single, on=key, how="left")
    chosen_geom = []
    for r in base.itertuples():
        gd = getattr(r, "geom_D", None)
        gc = getattr(r, "geom_C", None)
        chosen_geom.append(gd if gd is not None else gc)

    # -------- LEAN layer (avoid duplicate key columns on right) --------
    # Only non-key columns here to keep column labels unique
    lean_cols = [
        "reach_id_raw","source_version","source_crid",
        "sword_version_assumed","source_dir","source_file","crosswalk_status"
    ]
    df_for_lean = df_combined.copy()
    for c in lean_cols:
        if c not in df_for_lean.columns:
            df_for_lean[c] = None

    # Right side must not duplicate key labels:
    lean_nonkeys = [c for c in lean_cols if c not in key]
    right_lean = df_for_lean.loc[:, key + lean_nonkeys].drop_duplicates(subset=key)

    g_lean = gpd.GeoDataFrame(base[key], geometry=chosen_geom, crs="EPSG:4326") \
             .merge(right_lean, on=key, how="left", validate="one_to_one")

    # -------- FULL layer (ALL attributes from winner rows) --------
    # df_combined is already unique per key by construction in Block 7
    right_full = df_combined.drop_duplicates(subset=key)
    g_full = gpd.GeoDataFrame(base[key], geometry=chosen_geom, crs="EPSG:4326") \
            .merge(right_full, on=key, how="left", validate="one_to_one")

    ensure_dir(OUT_GPKG)
    g_lean.to_file(OUT_GPKG, layer="combined_reaches", driver="GPKG")
    g_full.to_file(OUT_GPKG, layer="combined_reaches_full", driver="GPKG")
    log(f"Wrote: {OUT_GPKG} (layers: combined_reaches, combined_reaches_full)")


Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\combined_reaches.gpkg (layers: combined_reaches, combined_reaches_full)


In [54]:
# =========================
# Block 10 — Console summary
# =========================
print("\n=== Crossing v2c summary ===")
print(f"Rows kept — C: {len(gC_tbl)}, D: {len(gD_tbl)}, Combined (unique): {len(df_combined)}")
print(f"Stations seen in combined: {df_combined['station_id'].nunique()}")
print("Outputs:")
print(f" - {OUT_COMBINED_CSV}")
print(f" - {OUT_PER_C}")
print(f" - {OUT_PER_D}")
print(f" - {OUT_RESOLVER_SUMMARY}")
print(f" - {OUT_DEDUPE}")
if not unmapped.empty:
    print(f" - {OUT_UNMAPPED}  [review recommended]")
if WRITE_GPKG:
    print(f" - {OUT_GPKG}::combined_reaches")
    print(f" - {OUT_GPKG}::combined_reaches_full")


=== Crossing v2c summary ===
Rows kept — C: 846, D: 151, Combined (unique): 642
Stations seen in combined: 11
Outputs:
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\combined_timeseries.csv
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\per_version_C.csv
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\per_version_D.csv
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\id_resolver_summary.csv
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\dedupe_log.csv
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\unmapped_rows.csv  [review recommended]
 - C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate

In [55]:
# Write README_v2c.txt into the output folder
from pathlib import Path
from datetime import datetime, timezone
BASE = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge")
OUT_DIR = BASE / r"output\1_Crossing_v2"

readme = f"""C2 v2c — Crossing Outputs (README)
Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} (local)

Purpose
-------
This package contains the Version-aware crossing between SWOT RiverSP reaches and target stations,
with canonical reach IDs and a unified time series. Version D (SWORD v17b) is the canonical ID
space; Version C (SWORD v16) is mapped into D IDs per station. For each (station_id, reach_id_canon,
obs_time), if both C and D exist, D wins.

Inputs (paths)
--------------
- Stations (AOI only): input\\Stations\\PatriciaTEAMS\\Round_3_Process\\All_stations_Rwanda_WaterPortal.shp
- RiverSP reaches:     input\\SWOT_L2_HR_RiverSP\\versionC\\reach\\**\\*.shp
                       input\\SWOT_L2_HR_RiverSP\\versionD\\reach\\**\\*.shp
- Crosswalk (C3):      output\\diagnosis\\version_id_crosswalk.csv  (or overlap_matches.csv fallback)

Key Parameters
--------------
- AOI buffer around stations: 50 km (bbox prefilter on files)
- Read mode: streaming (no full in-memory loads)
- Canonicalization: map C_top_id → D_top_id per station (from crosswalk)
- Precedence: prefer D over C for identical (station_id, reach_id_canon, obs_time)
- CRID detection: robust (e.g., PGC0_01 / PIC2_01 / PID0_03 from filename or folder)
- SWORD version tag: v16 for Version C, v17b for Version D (assumed)

Outputs (this folder)
---------------------
- combined_timeseries.csv        One row per (station_id, reach_id_canon, obs_time); D-over-C winner row
- per_version_C.csv              All C rows that matched target IDs (ALL reach attributes)
- per_version_D.csv              All D rows that matched target IDs (ALL reach attributes)
- id_resolver_summary.csv        Per-station counts and crosswalk summary
- dedupe_log.csv                 Rows dropped when D overwrote C for the same key
- unmapped_rows.csv              C rows that didn’t map to a D ID (if any)
- combined_reaches.gpkg
  • layer: combined_reaches       (Lean: keys + provenance + geometry)
  • layer: combined_reaches_full  (Full: ALL attributes from the winning row + geometry)

Schema — Key Columns (present in CSVs and/or GPKG)
--------------------------------------------------
Keys & provenance:
- station_id              Station identifier (string)
- reach_id_raw            ID as read from the source file
- reach_id_canon          Canonical reach ID (D space); equals reach_id_raw for D rows and mapped C rows
- obs_time                Observation time (string from time_str/time_tai/time)
- source_version          'C' or 'D'
- source_crid             Processing CRID token parsed from filename or folder (e.g., PGC0_01, PIC2_01, PID0_03)
- sword_version_assumed   'v16' for C, 'v17b' for D
- source_dir, source_file Source granule path components
- crosswalk_status        'stable' (D), 'C→D' (mapped), 'unmapped' (C not mapped)

All reach attributes:
- The full set of properties present in the RiverSP reach shapefile is carried forward verbatim:
  • Discharges: consensus & per-algorithm families (e.g., dschg_c, dschg_c_u, dschg_m, dschg_b, dschg_h,
    dschg_s, dschg_i, and gauge-constrained variants where present; plus associated *_q, *_q_b, *_sf, *_u).
  • WSE / width / slope / area suites (e.g., wse, wse_u; width, width_u/width_c; slope, slope_u; area_total …).
  • Coverage & quality context (e.g., reach_q, reach_q_b, partial_f, dark_frac, obs_frac_n, n_good_nod, ice_*).
  • Timing / reference fields as provided (time_str, time_tai/time; p_lat, p_lon; geoid_*; and other diagnostics
    if present in the source DBF).
- Note: attribute names reflect the original shapefile fields (10-char truncation may apply upstream).

Interpretation Notes
--------------------
- D-over-C: For a given station×reach×time, if both versions exist, the D record is kept with its own attributes.
- Missing discharge values: If discharges are fill (e.g., -999 or NaN), consult reach_q, reach_q_b and
  discharge quality flags (e.g., dschg_q_b) to understand the reason (coverage, slope sign, consensus limits, etc.).
- Uncertainties: Version-D release notes indicate uncertainty fields may not be fully validated yet; interpret *_u
  and *_sf with caution.
- ID changes: Many reaches legitimately changed ID between SWORD v16 and v17b; this script resolves per station via
  the C3 crosswalk and reports unmapped C rows.

Quick GIS Tips
--------------
- Symbolize quality: filter reach_q in (0,1) for “usable/suspect”; inspect reach_q_b bitflags for why a reach is degraded.
- Compare versions visually: combined_reaches_full contains the chosen geometry; to see C vs D shapes separately,
  inspect the per_version_C/D CSVs with their geometries in the raw RiverSP files if needed.

Repro / How to Re-run
---------------------
- Run crossing_v2c with the same parameters. Outputs are written here:
  {OUT_DIR}
- To speed up or widen the scan, adjust:
  STATION_PREFILTER_BUFFER_KM (default 50), READ_WITHIN_AOI_ONLY (default True).

Change Log
----------
- v2c: carry ALL attributes, two GPKG layers (lean + full), robust CRID, streaming scan, D-over-C unique table.
- v2b: discharge/flags inheritance, CRID fix, one-to-one merges, memory-safe GPKG.
- v2: consolidated scanning + AOI + crosswalk assembly.
"""

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "README_v2c.txt").write_text(readme, encoding="utf-8")
print(f"Wrote: {OUT_DIR / 'README_v2c.txt'}")


Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\1_Crossing_v2\README_v2c.txt
